# NanoGPT (Learn)

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
import os
import sys
from pathlib import Path

from pathlib import Path

CWD = os.path.realpath(os.getcwd())
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

DATA_DIR = Path(PARENT_DIR).parent / 'data'

In [3]:
from reader.loader import TextDataset

dataset = TextDataset(DATA_DIR, device=device)

100%|██████████| 5/5 [00:00<00:00, 13469.18it/s]


In [4]:
from src.modules.architecture.ngram_lm import NgramLanguageModel
from reader.preprocess import decode
import torch

BATCH_SIZE = 16
EMBEDDING_SIZE = 64
SEQ_LENGTH = 32
DROPOUT_RATE = 0.2

N_HEADS = 4
N_BLOCKS = 4
LR = 1e-3

EVAL_ITER = 100
EVAL_INTERVAL = 100
EPOCH_SIZE = 5000

torch.manual_seed(1337)

model = NgramLanguageModel(vocab_size=dataset.vocab_size, n_heads=N_HEADS, embedding_size=EMBEDDING_SIZE, seq_length=SEQ_LENGTH, 
                            n_blocks=N_BLOCKS, dropout_rate=DROPOUT_RATE, device=device)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("Number of parameters:")
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

Number of parameters:
0.212825 M parameters


In [5]:
@torch.no_grad()
def estimate_loss(x, y, model):
    losses = torch.zeros(EVAL_ITER)
    for k in range(EVAL_ITER):
        logits, loss = model(x, y)
        losses[k] = loss.item()
    return losses.mean()

In [ ]:
for iter in range(EPOCH_SIZE):

    # every once in a while evaluate the loss on train and val sets
    if iter % EVAL_INTERVAL == 0 or iter == EPOCH_SIZE - 1:
        model.eval()
        x_train, y_train = dataset.load_train(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        x_val, y_val = dataset.load_test(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        train_losses = estimate_loss(x_train, y_train, model)
        val_losses = estimate_loss(x_val, y_val, model)
        print(f"step {iter}: train loss {train_losses:.4f}, val loss {val_losses:.4f}")
        model.train()

    # sample a batch of data
    x_train, y_train = dataset.load_train(BATCH_SIZE)

    # evaluate the loss
    logits, loss = model(x_train, y_train)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.6745, val loss 4.6668
step 100: train loss 2.5942, val loss 2.5948
step 200: train loss 2.4359, val loss 2.4038
step 300: train loss 2.2259, val loss 2.3016
step 400: train loss 2.1360, val loss 2.1561
step 500: train loss 2.0047, val loss 2.0626
step 600: train loss 1.9620, val loss 2.0441
step 700: train loss 1.8821, val loss 1.9523
step 800: train loss 1.9000, val loss 1.9562
step 900: train loss 1.8300, val loss 1.9003
step 1000: train loss 1.7681, val loss 1.8542
step 1100: train loss 1.7794, val loss 1.8455
step 1200: train loss 1.6692, val loss 1.8741
step 1300: train loss 1.6611, val loss 1.8483
step 1400: train loss 1.6763, val loss 1.8283
step 1500: train loss 1.6832, val loss 1.7707
step 1600: train loss 1.6908, val loss 1.7833
step 1700: train loss 1.6372, val loss 1.8060
step 1800: train loss 1.6577, val loss 1.7580
step 1900: train loss 1.5862, val loss 1.7574
step 2000: train loss 1.5768, val loss 1.7258
step 2100: train loss 1.5193, val loss 1.7666


In [ ]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

In [ ]:
from reader.preprocess import encode
context = torch.tensor([encode(dataset.stoi, 'fyodor')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))